In [8]:
class settings:
  def __init__(self):
    self.iter=10
    self.results_dir="experiment_main100"
    self.dataset = "MIMIC"
    self.metrics = ["AUC","AUPRC", "F1","PPV","Sensitivity", "Specificity", "NPV", "MCC", "Accuracy"]
    self.metrics_wilcoxon = ["AUC", "F1","Sensitivity", "Specificity", "MCC"]
    self.methods = ['STATIC+TF-IDF' ,'STATIC+BOW', 'STATIC+BiPadLSTM','STATIC+Med2Vec','STATIC+Dipole',
                    'STATIC+CEHR-BERT','STATIC+DOME','STATIC+GRU']

args = settings()

In [9]:
import pickle
# Caricamento dei risultati
results = {m: {} for m in args.methods}

for i in range(args.iter):
    for m in args.methods:
        meth = '_'.join(m.split('+'))
        path = (f"{args.results_dir}/results_{args.dataset}_{meth}_ITER{i}.pkl")
        with open(path, "rb") as f:
            results[m][i] = pickle.load(f)

In [10]:
import pandas as pd
dfs = {}
for metric in args.metrics:
    values = {method : [] for method in args.methods}
    for method in args.methods:
        lista = []
        for iter in range(args.iter):
            lista += results[method][iter][metric]
        values[method] = lista
    dfs[metric] = pd.DataFrame.from_dict(values)


In [11]:
import pandas as pd
from scripts.stats import friedman_test, wilcoxon_test, wilcoxon_ranking

alpha = 0.05
friedman_results = {}
wilcoxon_p_matrices = {}
wilcoxon_rankings = {}

for metric in args.metrics:
    df = dfs[metric]

    if "STATIC" in df.columns:
        df = df.drop(columns=["STATIC"])

    stat, p = friedman_test(df, alpha=alpha, debug=True)

    friedman_results[metric] = {
        "statistic": stat,
        "p_value": p,
        "global_diff": "YES" if p < alpha else "NO"
    }

    if p < alpha:
        p_matrix, reject, p_corr, results = wilcoxon_test(
            df,
            alpha=alpha,
            debug=True
        )

        ranking = wilcoxon_ranking(
            df,
            reject,
            higher_is_better=True
        )

        wilcoxon_p_matrices[metric] = p_matrix
        wilcoxon_rankings[metric] = ranking

friedman_df = pd.DataFrame(friedman_results).T


=== FRIEDMAN TEST ===
Statistic: 93.0638
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 79.8664
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 42.9571
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 12.2347
p-value: 0.093099
➡️ No global differences

=== FRIEDMAN TEST ===
Statistic: 70.5452
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 80.7449
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 45.4899
p-value: 0.000000
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 26.5172
p-value: 0.000407
➡️ SIGNIFICANT global differences

=== FRIEDMAN TEST ===
Statistic: 45.4215
p-value: 0.000000
➡️ SIGNIFICANT global differences


In [12]:
friedman_latex = friedman_df.copy()

friedman_latex["p_value"] = friedman_latex["p_value"].apply(
    lambda x: f"{x:.2e}"
)

friedman_latex["global_diff"] = friedman_latex["global_diff"].map({
    "YES": r"\checkmark",
    "NO": r"\ding{55}"
})

friedman_latex = friedman_latex.rename(columns={
    "statistic": "Statistic",
    "p_value": "$p$-value",
    "global_diff": "Difference"
})

friedman_latex = friedman_latex.T
latex_table = friedman_latex.T.to_latex(
    escape=False,
    caption="Friedman test results for all evaluation metrics.",
    label="tab:friedman",
    column_format="l" + "c"*len(friedman_latex.T.columns)
)

print(latex_table)

\begin{table}
\caption{Friedman test results for all evaluation metrics.}
\label{tab:friedman}
\begin{tabular}{lccc}
\toprule
 & Statistic & $p$-value & Difference \\
\midrule
AUC & 93.063825 & 2.90e-17 & \checkmark \\
AUPRC & 79.866365 & 1.47e-14 & \checkmark \\
F1 & 42.957132 & 3.40e-07 & \checkmark \\
PPV & 12.234734 & 9.31e-02 & \ding{55} \\
Sensitivity & 70.545166 & 1.15e-12 & \checkmark \\
Specificity & 80.744940 & 9.71e-15 & \checkmark \\
NPV & 45.489947 & 1.10e-07 & \checkmark \\
MCC & 26.517233 & 4.07e-04 & \checkmark \\
Accuracy & 45.421455 & 1.13e-07 & \checkmark \\
\bottomrule
\end{tabular}
\end{table}



In [13]:
wilcoxon_rankings.keys()

dict_keys(['AUC', 'AUPRC', 'F1', 'Sensitivity', 'Specificity', 'NPV', 'MCC', 'Accuracy'])

In [14]:
ranking_table = pd.DataFrame()

for metric in args.metrics_wilcoxon:

    ranking = wilcoxon_rankings[metric].copy()

    ranking["rank"] = ranking["score"].rank(
        method="min",
        ascending=False
    ).astype(int)

    ranking_table[metric] = ranking["rank"]

ranking_table = ranking_table[args.metrics_wilcoxon]

# ordinamento gerarchico:
ranking_table = ranking_table.sort_values(
    by=args.metrics_wilcoxon,
    ascending=True      # rank 1 migliore
)
print(ranking_table.to_latex())

\begin{tabular}{lrrrrr}
\toprule
 & AUC & F1 & Sensitivity & Specificity & MCC \\
\midrule
STATIC+DOME & 1 & 1 & 1 & 6 & 1 \\
STATIC+TF-IDF & 2 & 4 & 3 & 6 & 5 \\
STATIC+BiPadLSTM & 3 & 2 & 2 & 8 & 2 \\
STATIC+BOW & 4 & 8 & 7 & 2 & 5 \\
STATIC+CEHR-BERT & 5 & 4 & 6 & 4 & 5 \\
STATIC+Med2Vec & 5 & 7 & 7 & 1 & 5 \\
STATIC+Dipole & 7 & 3 & 4 & 2 & 2 \\
STATIC+GRU & 8 & 4 & 5 & 5 & 2 \\
\bottomrule
\end{tabular}

